# Донавчання YOLO: люди і техніка з дрона (тепловізор / RGB)

1. **Середовище → Змінити тип середовища → GPU (T4)**.
2. Виконуйте комірки по черзі.
3. `USE_DRIVE = True` зберігає дані й результати на Google Диску, щоб навчання можна було продовжити після обриву сесії. Colab тоді попросить дозвіл на доступ до Диска.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Файли навчання
Комірка попросить завантажити `training_kit.zip`.

In [ ]:
USE_DRIVE = False
import os, zipfile
from google.colab import files
WORK = '/content/sar_training'
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK = '/content/drive/MyDrive/sar_training'
os.makedirs(WORK, exist_ok=True)
if not os.path.exists(f'{WORK}/train.py'):
    up = files.upload()                          # оберіть training_kit.zip
    zipfile.ZipFile(next(iter(up))).extractall('/content/kit')
    !cp -r /content/kit/training/. "{WORK}/"
%cd "{WORK}"
!ls

In [ ]:
!pip install -q ultralytics ncnn pnnx

## 2. Власні кадри (необов'язково, але дуже бажано)
Розмічені кадри у форматі YOLO (класи `0 = person`, `1 = vehicle`) кладуть у
`own_data/thermal/images` + `labels` або `own_data/rgb/images` + `labels`.
Кадри без об'єктів теж корисні: вони зменшують кількість хибних тривог.

## 3. Модель для тепловізора

In [ ]:
!python prepare_data.py --target thermal          # HIT-UAV + VisDrone (у сірому) + ваші кадри

In [ ]:
# YOLO11s, ліміт 1.5 год (кількість епох підбирається під час). Після обриву: --resume
!python train.py --target thermal --model yolo11s.pt --epochs 60 --time 1.5

In [ ]:
# !python train.py --target thermal --resume

## 4. Порівняння зі стандартною моделлю на тестових кадрах HIT-UAV

In [ ]:
!python compare.py --data datasets/sar_thermal.yaml --models yolo11n.pt trained/sar_thermal.pt --imgsz 640 --out trained/compare.txt

## 5. Модель для RGB-камери (необов'язково)

In [ ]:
# !python prepare_data.py --target rgb
# !python train.py --target rgb --model yolo11s.pt --epochs 60 --time 1.5

## 6. Результат для Raspberry Pi

In [ ]:
!cat trained/*_metrics.txt
!cd trained && zip -qr /content/models_for_pi.zip .
files.download('/content/models_for_pi.zip')

Розпакуйте архів у папку `thermal_viewer` на Pi і пропишіть у `config.env`:
```
ARGS="--ncnn --imgsz 416 --thermal-model sar_thermal.pt"
```
Для RGB-моделі додайте `--model sar_rgb.pt`. Потім виконайте `sudo systemctl restart thermal-viewer`.